In [ ]:
# 合并 SMPS 与 APS 数据，计算总的粒径分布
# 包括以下步骤：
# 1. 导入库和函数
# 2. 读取 SMPS 和 APS 数据

In [1]:
## 1. 导入库和函数
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def read_one_smps_export(path: Path, encoding="cp1252", bin_start_idx=9, bin_end_idx=112, time_format='%Y/%m/%d %H:%M:%S', scan_lines=100) -> pd.DataFrame:
    '''
    读取由 SMPS 导出的单个 TXT 文件, 返回包含时间和粒径列的 DataFrame.

    参数:
    '''
    # 1) 定位表头行 (只扫描前`scan_lines`行)
    with open(path, "r", encoding=encoding, errors="replace") as f:
        lines = []
        for _ in range(scan_lines):
            try:
                lines.append(next(f))
            except StopIteration:
                break

    header_idx = None
    for i, line in enumerate(lines):
        if 'Units' in line:
            units_idx = i
        if "Sample #" in line and "Date" in line and "Start Time" in line:
            if line.count("\t") >= 5 or line.count(",") >= 5 or line.count(";") >= 5:
                header_idx = i
                break
    if header_idx is None:
        raise ValueError(f"{path.name}: 扫描前`scan_lines`行, 未找到表头行")
    
    header_line = lines[header_idx]
    #print(f"{path.name}的数据类型： {lines[units_idx].split('\t')[1]} ")
    if lines[units_idx].split('\t')[1].strip() != "dw/dlogDp":
        raise ValueError(f"{path.name}: 数据类型不是 dN/dlogDp，而是 {lines[units_idx].split('\t')[1]}")

    # 2) 识别分隔符
    delims = {"\t": header_line.count("\t"), ",": header_line.count(","), ";": header_line.count(";")}
    sep = max(delims, key=delims.get)

    # 3) 从表头行开始读
    df = pd.read_csv(path, sep=sep, encoding=encoding, skiprows=0, header=header_idx, engine="python")

    # 4) 生成 datetime
    t = pd.to_datetime(df["Date"].astype(str).str.strip() + " " + df["Start Time"].astype(str).str.strip(),
                       errors="coerce", # 如果需要更严格的解析, 可以去掉此行
                       format=time_format)
    bad = t.isna()
    if bad.any():
        print("解析失败行数:", bad.sum())
        print("如果行数较多，可能需要调整 time_format 参数")
        print(df.loc[bad, ["Date", "Start Time"]].head(10))
        # 需要的话：raise ValueError("Datetime parsing failed")

    df = df.loc[t.notna()].copy()
    df["__dt__"] = t.loc[t.notna()].values

    # 5) 粒径列
    BIN_COLS = list(df.columns[bin_start_idx:bin_end_idx])
    # 判断粒径列是否正确选取
    try:
        _ = np.array(BIN_COLS).astype(float)
    except ValueError:
        raise ValueError(f"{path.name}: 粒径列名不能转换为浮点数，可能选取错误：{BIN_COLS[:5]}...\n请尝试调整 bin_start_idx 和 bin_end_idx 参数")

    out = df[["__dt__"] + BIN_COLS].copy()
    out["__file__"] = path.name  # 便于追踪来源
    return out


def read_one_aps_export(path: Path, encoding="cp1252", bin_start_idx=5, bin_end_idx=56, time_format="%m/%d/%y %H:%M:%S", scan_lines=100) -> pd.DataFrame:
    '''
    读取由 APS 导出的单个 TXT 文件, 返回包含时间和粒径列的 DataFrame.

    参数:
    '''
    # 1) 定位表头行 (只扫描前`scan_lines`行)s
    with open(path, "r", encoding=encoding, errors="replace") as f:
        lines = []
        for _ in range(scan_lines):
            try:
                lines.append(next(f))
            except StopIteration:
                break

    header_idx = None
    for i, line in enumerate(lines):
        if "Sample #" in line and "Date" in line and "Start Time" in line:
            if line.count("\t") >= 5 or line.count(",") >= 5 or line.count(";") >= 5:
                header_idx = i
                break
    if header_idx is None:
        raise ValueError(f"{path.name}: 扫描前`scan_lines`({scan_lines})行, 未找到表头行")

    header_line = lines[header_idx]

    # 2) 识别分隔符
    delims = {"\t": header_line.count("\t"), ",": header_line.count(","), ";": header_line.count(";")}
    sep = max(delims, key=delims.get)

    # 3) 从表头行开始读
    df = pd.read_csv(path, sep=sep, encoding=encoding, skiprows=header_idx, header=0, engine="python")
    #print(f"`{path.name}`的数据类型: {df.iloc[0, 3]}")
    if df.iloc[0, 3] != "dN/dlogDp":
        raise ValueError(f"{path.name}: 数据类型不是 dN/dlogDp，而是 {df.iloc[0, 3]}")
    # 4) 生成 datetime
    t = pd.to_datetime(df["Date"].astype(str).str.strip() + " " + df["Start Time"].astype(str).str.strip(),
                       errors="coerce", # 如果需要更严格的解析, 可以去掉此行
                       format=time_format)
    bad = t.isna()
    if bad.any():
        print("解析失败行数:", bad.sum())
        print("如果行数较多，可能需要调整 time_format 参数")
        print(df.loc[bad, ["Date", "Start Time"]].head(10))
        # 需要的话：raise ValueError("Datetime parsing failed")
    
    df = df.loc[t.notna()].copy()
    df["__dt__"] = t.loc[t.notna()].values

    # 5) 粒径列 
    BIN_COLS = list(df.columns[bin_start_idx:bin_end_idx])# Note: 这种方法舍去了'<0.523'这一列, 缺失部分由 SMPS 补全.
    # 判断粒径列是否正确选取
    try:
        _ = np.array(BIN_COLS).astype(float)
    except ValueError:
        raise ValueError(f"{path.name}: 粒径列名不能转换为浮点数，可能选取错误：{BIN_COLS[:5]}...\n请尝试调整 bin_start_idx 和 bin_end_idx 参数")

    out = df[["__dt__"] + BIN_COLS].copy()
    out["__file__"] = path.name  # 便于追踪来源
    return out

def dups_check(df: pd.DataFrame, time_col="__dt__") -> pd.DataFrame:
    '''
    检查 DataFrame 中的重复时间戳, 若有重复, 则尝试去重.
    '''
    n_dups = df.duplicated(subset=time_col).sum()
    if n_dups > 0:
        print(f"⚠️ 数据中存在 {n_dups} 行重复时间戳，尝试去重:")
        n_before = len(df)
        df = df.drop_duplicates(subset=time_col, keep="last").reset_index(drop=True)# 去重策略：同一时间点保留“最后出现的那条”（通常后导出的文件更完整）
        print("\n=== 去重结果 ===")
        print("合并前总行数 =", n_before)
        print("重复时间点数 =", int(n_dups))
        print("去重后总行数 =", len(df))
    else:
        print("✅ 数据中没有重复时间戳")

def diff_check(df: pd.DataFrame, time_col="__dt__") -> None:
    '''
    计算采样间隔, 并打印基本分布情况, 以帮助诊断数据质量问题.
    '''
    dt_diff = df[time_col].diff().dropna().dt.total_seconds()
    print("采样间隔(秒)：中位数 =", float(dt_diff.median()),
        "最小 =", float(dt_diff.min()),
        "最大 =", float(dt_diff.max()))

In [2]:
## 2. 读取 SMPS 和 APS 数据
### 1) 读取 APS 数据
aps_dir = Path(r"D:\Coding\Data\Lanzhou_aerosol\APS_dNdlogDp")
aps_data = list(aps_dir.glob('*.TXT'))
print("APS 文件数 =", len(aps_data))

aps_all = []
for p in aps_data:
    try:
        one = read_one_aps_export(p)
        aps_all.append(one)
        print(f"✅ {p.name}: {len(one)} 行, 时间 {one['__dt__'].min()} ~ {one['__dt__'].max()}")
    except Exception as e:
        print(f"❌ {p.name}: 读取失败 -> {repr(e)}")

aps_df = pd.concat(aps_all, ignore_index=True)
aps_df = aps_df.sort_values("__dt__").reset_index(drop=True)

dups_check(aps_df)# 检查 APS 数据中的重复时间戳
diff_check(aps_df)# 检查 APS 数据中的采样间隔

print("\n------------------------------------------------------------\n")

### 2) 读取 SMPS 数据
smps_dir = Path(r"D:\Coding\Data\Lanzhou_aerosol\SMPS_dNdlogDp")
smps_data = list(smps_dir.glob('*.TXT'))
print("SMPS 文件数 =", len(smps_data))

smps_all = []
for p in smps_data:
    try:
        one = read_one_smps_export(p)
        smps_all.append(one)
        print(f"✅ {p.name}: {len(one)} 行, 时间 {one['__dt__'].min()} ~ {one['__dt__'].max()}")
    except Exception as e:
        print(f"❌ {p.name}: 读取失败 -> {repr(e)}")

smps_df = pd.concat(smps_all, ignore_index=True)
smps_df = smps_df.sort_values("__dt__").reset_index(drop=True)

dups_check(smps_df)# 检查 SMPS 数据中的重复时间戳
diff_check(smps_df)# 检查 SMPS 数据中的采样间隔

APS 文件数 = 11
✅ 20241206.TXT: 123 行, 时间 2024-12-06 13:42:24 ~ 2024-12-06 23:57:34
✅ 20241208-1230.TXT: 6493 行, 时间 2024-12-08 00:04:48 ~ 2024-12-30 18:07:44
✅ 20241230.TXT: 46 行, 时间 2024-12-30 18:33:41 ~ 2024-12-30 22:21:08
✅ 20241231-20250119.TXT: 5445 行, 时间 2024-12-31 11:01:15 ~ 2025-01-19 12:55:29
✅ 20250119.TXT: 96 行, 时间 2025-01-19 14:20:32 ~ 2025-01-19 22:20:23
✅ 20250120-0212.TXT: 6598 行, 时间 2025-01-20 19:24:36 ~ 2025-02-12 22:19:20
✅ 20250215-0221.TXT: 1827 行, 时间 2025-02-15 12:50:21 ~ 2025-02-21 22:22:52
✅ 20250222-0313.TXT: 5360 行, 时间 2025-02-22 20:23:25 ~ 2025-03-13 11:31:28
✅ 20250313-0530.TXT: 22296 行, 时间 2025-03-13 11:52:35 ~ 2025-05-30 15:06:08
✅ 20250911-0917.TXT: 4382 行, 时间 2025-09-11 16:20:23 ~ 2025-09-17 18:22:23
✅ 20250917-0929.TXT: 8753 行, 时间 2025-09-17 18:31:40 ~ 2025-09-29 22:15:40
✅ 数据中没有重复时间戳
采样间隔(秒)：中位数 = 302.0 最小 = 119.0 最大 = 8990055.0

------------------------------------------------------------

SMPS 文件数 = 33
✅ 20241205-1215.TXT: 7440 行, 时间 2024-12-05 13:17:56 

In [ ]:
# ===== 剔除异常浓度时间段 =====
smps_bad_periods = [
    ("2024-12-14 23:00", "2024-12-16 00:00"),
    ("2024-12-22 19:00", "2024-12-23 12:00"),
    ("2025-03-10 04:00", "2025-03-10 14:00"),
    ("2025-03-25 21:00", "2025-03-25 23:00"),
    ("2025-05-30 15:10", "2025-05-30 15:15"),# 扫描不完全导致浓度异常低, 详情见SMPS原始数据文件
    ("2025-06-19 00:00", "2025-06-20 00:00"),
    ("2025-07-23 14:50", "2025-07-23 15:00"),
    ("2025-08-11 17:49", "2025-08-11 17:50"),
    ("2025-08-14 23:00", "2025-08-15 16:00"),
    ("2025-10-16 23:00", "2025-10-19 00:00"),
]
for start_str, end_str in smps_bad_periods:
    start = pd.to_datetime(start_str)
    end = pd.to_datetime(end_str)
    mask = (smps_df["__dt__"] >= start) & (smps_df["__dt__"] <= end)
    n_bad = mask.sum()
    if n_bad > 0:
        print(f"剔除 SMPS 数据中 {start} ~ {end} 的 {n_bad} 行")
        smps_df = smps_df.loc[~mask].reset_index(drop=True)

剔除 SMPS 数据中 2024-12-14 23:00:00 ~ 2024-12-16 00:00:00 的 749 行


剔除 SMPS 数据中 2024-12-22 19:00:00 ~ 2024-12-23 12:00:00 的 510 行
剔除 SMPS 数据中 2025-03-10 04:00:00 ~ 2025-03-10 14:00:00 的 300 行
剔除 SMPS 数据中 2025-03-25 21:00:00 ~ 2025-03-25 23:00:00 的 52 行
剔除 SMPS 数据中 2025-05-30 15:10:00 ~ 2025-05-30 15:15:00 的 2 行
剔除 SMPS 数据中 2025-06-19 00:00:00 ~ 2025-06-20 00:00:00 的 100 行
剔除 SMPS 数据中 2025-07-23 14:50:00 ~ 2025-07-23 15:00:00 的 5 行
剔除 SMPS 数据中 2025-08-14 23:00:00 ~ 2025-08-15 16:00:00 的 510 行
剔除 SMPS 数据中 2025-10-16 23:00:00 ~ 2025-10-19 00:00:00 的 1184 行


In [4]:
## 统一时间轴: 重采样并取平均值

# 定义重采样的时间频率，建议设为 10min (与 INP 采样时间一致)
resample_freq = '10min'

# 设置时间为索引
smps_df.set_index("__dt__", inplace=True)
aps_df.set_index("__dt__", inplace=True)

# 移除不需要合并的辅助列
if "__file__" in smps_df.columns:
    smps_df = smps_df.drop(columns=["__file__"])
if "__file__" in aps_df.columns:
    aps_df = aps_df.drop(columns=["__file__"])

print(f"正在按 {resample_freq} 进行重采样并取平均值...")

# 对 SMPS 和 APS 分别重采样
# closed='left', label='left' 表示时间戳代表该窗口的起始时间
smps_resampled = smps_df.resample(resample_freq, closed='left', label='left').mean()
aps_resampled = aps_df.resample(resample_freq, closed='left', label='left').mean()

print(f"SMPS 重采样后行数: {len(smps_resampled)}")
print(f"APS 重采样后行数: {len(aps_resampled)}")

# 检查一下是否有全空行（由于原始数据中有大段空白时间段导致的）
smps_resampled.dropna(how='all', inplace=True)
aps_resampled.dropna(how='all', inplace=True)

print(f"剔除全空行后 - SMPS: {len(smps_resampled)}, APS: {len(aps_resampled)}")

正在按 10min 进行重采样并取平均值...
SMPS 重采样后行数: 45419
APS 重采样后行数: 42820
剔除全空行后 - SMPS: 36563, APS: 26976


In [5]:
## 转换粒径列名为数值并排序

def clean_and_sort_columns(df):
    # 找出所有可以转换为数字的列（排除非粒径列，如果有的话）
    # 在重采样后，现在的列应该全都是粒径值了
    new_cols = []
    for col in df.columns:
        try:
            new_cols.append(float(col))
        except ValueError:
            # 如果有不能转成数字的列，保持原样（虽然通常不应该有）
            new_cols.append(col)
    
    df.columns = new_cols
    # 按列名（粒径大小）从小到大排序
    df = df.reindex(sorted(df.columns), axis=1)
    return df

smps_resampled = clean_and_sort_columns(smps_resampled)
print(f"SMPS 粒径范围: {min(smps_resampled.columns)} ~ {max(smps_resampled.columns)} nm")

# 注意：APS 的单位通常是微米(μm)，如果你的 SMPS 是纳米(nm)，
# 我们需要统一单位。通常统一为纳米 (nm)。
# 假设 SMPS 列名是 10~1000 左右，APS 列名是 0.5~20 左右。
# 我们检查一下：
aps_cols_sample = float(aps_resampled.columns[0])
if aps_cols_sample < 20: # 如果数值很小，说明是微米
    print("检测到 APS 单位为微米 (μm)，正在转换为纳米 (nm)...")
    aps_resampled.columns = [float(c) * 1000 for c in aps_resampled.columns]

aps_resampled = clean_and_sort_columns(aps_resampled)

print(f"APS 粒径范围: {min(aps_resampled.columns)} ~ {max(aps_resampled.columns)} nm")

SMPS 粒径范围: 13.1 ~ 532.8 nm
检测到 APS 单位为微米 (μm)，正在转换为纳米 (nm)...
APS 粒径范围: 542.0 ~ 19810.0 nm


In [6]:
## 将 APS 的空气动力学直径 Da 转换为电迁移率直径 Dm

def convert_da_to_dm(da, rho_eff=1.5, rho_0=1.0, x=1.0, slip_correction=False):
    """
    将空气动力学直径 Da 转换为电迁移率直径 Dm

    da: APS 测得的空气动力学直径 (数组或标量，单位 um)

    rho_eff: 假设的有效密度 (g/cm3)

    rho_0: 标准密度 (1.0 g/cm3)

    x: 形状修正因子 (shape factor = 1.0, 表示球形粒子)
    """
    if slip_correction:
        print ('At sizes of the APS–SMPS overlap size range the slip correction can be neglected;'
        ' for the particle density of 2 g cm−3, the shape factor of 1,'
        ' and the physical diameter of 500 nm the error in calculating the aerodynamic diameter is 4%.' \
        ' At lower particle densities and larger shape factors the error will be even smaller.')

        print('\nCunningham 滑移修正可忽略不计, 因此直接使用简单的 Da-Dm 关系进行转换')
        dm = da * np.sqrt(rho_0 * x / rho_eff)
    else:
        dm = da * np.sqrt(rho_0 * x / rho_eff)
    
    return dm

# APS 测得的粒径序列(单位 μm)
aps_da = aps_resampled.columns.values

# Da 转 Dm
rho_p = 1.5 # 假设气溶胶粒子的有效密度为 1.5 g/cm3
aps_dm = convert_da_to_dm(aps_da, rho_eff=rho_p)

# 创建一个新的 DataFrame 存储转换后的 APS 数据
aps_dm_resampled = aps_resampled.copy()
aps_dm_resampled.columns = aps_dm # 将列名替换为 Dm

# 输出结果
print(f"假设有效密度为: {rho_p} g/cm3")
print("-" * 40)
print(f"{'Aerodynamic Da (um)':<20} | {'Mobility Dm (um)':<20}")
print("-" * 40)
for a, m in zip(aps_da, aps_dm):
    print(f"{a:<20.4f} | {m:<20.4f}")

# 去除 APS 中大于 2500 nm 的粒径（为保证气溶胶数据与 INP 数据的一致性）
aps_dm_resampled = aps_dm_resampled.loc[:, aps_dm_resampled.columns <= 2500]
print(f"\n转换后 APS 粒径范围(已舍去 > 2500 nm): {min(aps_dm_resampled.columns)} ~ {max(aps_dm_resampled.columns)} nm")


假设有效密度为: 1.5 g/cm3
----------------------------------------
Aerodynamic Da (um)  | Mobility Dm (um)    
----------------------------------------
542.0000             | 442.5411            
583.0000             | 476.0175            
626.0000             | 511.1269            
673.0000             | 549.5022            
723.0000             | 590.3270            
777.0000             | 634.4178            
835.0000             | 681.7746            
898.0000             | 733.2139            
965.0000             | 787.9192            
1037.0000            | 846.7070            
1114.0000            | 909.5772            
1197.0000            | 977.3464            
1286.0000            | 1050.0146           
1382.0000            | 1128.3983           
1486.0000            | 1213.3139           
1596.0000            | 1303.1285           
1715.0000            | 1400.2916           
1843.0000            | 1504.8032           
1981.0000            | 1617.4797           
2129.0000          

In [7]:
def merge_smps_aps(smps_df, aps_df):
    """
    拼接 SMPS 和转换后的 APS 数据, 考虑到 APS 的前几个通道效率较低, 因此两仪器的重叠部分以 SMPS 为准.

    确保两者拥有相同的时间索引 (index), 并且列名（粒径通道）从小到大排序, 形成一个连续的粒径谱图.
    """
    # 找到 SMPS 的最大的粒径
    smps_max_Dp = smps_df.columns.max() # 应该是 532.8 nm

    # 仅保留 APS 大于 SMPS 最大粒径(通道)的部分
    aps_dm_filtered = aps_df.loc[:, aps_df.columns > smps_max_Dp]

    print(f"裁切后, APS 保留的最小粒径为: {aps_dm_filtered.columns.min():.2f} nm")
    
    # 按时间 (index) 拼接两个 DataFrame
    combined_df = pd.concat([smps_df, aps_dm_filtered], axis=1, join='outer')
    
    # 4. 对列名（粒径通道）进行升序排序，确保形成一个连续的谱图
    combined_df = combined_df.reindex(sorted(combined_df.columns), axis=1)
    
    return combined_df

# 执行拼接
final_psd_df = merge_smps_aps(smps_resampled, aps_dm_resampled)

# 检查结果
print(f"最终合并数据的维度 (行, 列): {final_psd_df.shape}")
print(f"最终合并数据的粒径范围: {final_psd_df.columns.min():.2f} nm ~ {final_psd_df.columns.max():.2f} nm")

final_psd_df.head()

裁切后, APS 保留的最小粒径为: 549.50 nm
最终合并数据的维度 (行, 列): (41175, 126)
最终合并数据的粒径范围: 13.10 nm ~ 2491.13 nm


,13.100000,13.600000,14.100000,14.600000,15.100000,15.700000,16.300000,16.800000,17.500000,18.100000,...,1303.128543,1400.291636,1504.803199,1617.479727,1738.321221,1868.144177,2006.948596,2157.183967,2318.033793,2491.131068
__dt__,,,,,,,,,,,,,,,,,,,,,
2024-12-05 13:10:00,NaN,3669.6795,3864.0485,3652.2275,4393.6500,4563.6845,5006.4640,4517.1175,5909.6450,5887.7560,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-12-05 13:20:00,NaN,680.0616,949.6386,1140.2944,1997.8640,2169.7248,2487.1176,2915.3344,3335.6998,3702.4604,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-12-05 13:30:00,NaN,729.7262,761.4722,1202.9416,1571.9470,2252.8378,2697.0230,2766.0006,3574.9622,3757.7100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-12-05 13:40:00,NaN,527.6346,798.8096,1049.8260,1362.6466,1631.7170,2233.7060,2606.7516,2856.9350,3733.1086,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-12-05 13:50:00,NaN,404.4900,713.0984,874.6400,1420.7614,1755.1516,2038.7658,2027.9900,3096.7948,3072.9496,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# 判断仪器运行状态

# 提取列名
cols_smps = [col for col in final_psd_df.columns if float(col) <= 533.0]
cols_aps  = [col for col in final_psd_df.columns if float(col) >= 549.0]

# 1. 计算每个时间点 NaN 的比例
nan_ratio_smps = final_psd_df[cols_smps].isna().sum(axis=1) / len(cols_smps)
nan_ratio_aps  = final_psd_df[cols_aps].isna().sum(axis=1) / len(cols_aps)

# 2. 判断运行状态：缺失率 <= 90% (即 0.9) 认为是运行状态 (True)
status_smps = nan_ratio_smps <= 0.90
status_aps  = nan_ratio_aps <= 0.90

# 3. 将状态合并到一个 DataFrame 中
status_df = pd.DataFrame({
    'Date': status_aps.index,
    'status_smps': status_smps,
    'status_aps': status_aps
})

In [9]:
def calculate_surface_area(df):
    """
    计算气溶胶总表面积浓度
    - 参数:
    df: pd.DataFrame, 其中, index为时间, columns为粒径通道(nm), value为 dN/dlogDp (#/cm^3)
    - 返回:
    pd.Series: 总表面积浓度 (单位: μm^2/cm^3)
    """
    
    # 1. 提取粒径通道(Dp)，将列名从字符串转为浮点数
    dp_nm = np.array(df.columns.astype(float))
    
    # 2. 将粒径单位从 nm 转换为 μm (为了得到常用的 μm^2/cm^3 单位)
    dp_um = dp_nm / 1000.0
    
    # 3. 计算每个通道的 dlogDp
    # 大多数仪器(如SMPS)的粒径在对数坐标下是等间距的
    # 我们通过计算相邻通道 log10(Dp) 的差值来获取
    log_dp = np.log10(dp_nm)
    
    # 计算相邻间距
    delta_log_dp = np.zeros_like(log_dp)
    # 对于中间部分，使用相邻通道的差值
    delta_log_dp[:-1] = np.diff(log_dp)
    # 最后一个通道的间距假设与倒数第二个相同
    delta_log_dp[-1] = delta_log_dp[-2]
    
    # 4. 计算每个通道的表面积贡献 dS
    # 公式: dS = pi * Dp^2 * (dN/dlogDp) * delta_log_dp
    # 注意: dp_um 是矢量, df 是矩阵, 利用广播机制计算
    
    # 计算每一项: pi * Dp^2 * dlogDp
    multiplier = np.pi * (dp_um**2) * delta_log_dp
    
    # 逐行相乘并求和
    # df.values 的形状是 (时间, 粒径), multiplier 的形状是 (粒径,)
    #surface_area_conc = (df.values * multiplier).sum(axis=1)
    surface_area_conc = df.multiply(multiplier, axis=1).sum(axis=1, min_count=1)
    
    # 5. 返回结果，保持与原数据相同的索引
    return pd.Series(surface_area_conc, index=df.index, name='Total_Surface_Area_um2_cm3')

# 计算
total_s = calculate_surface_area(final_psd_df)

print("\n计算得到的总表面积浓度 (μm^2/cm^3):")
print(total_s.dropna())


计算得到的总表面积浓度 (μm^2/cm^3):
__dt__
2024-12-05 13:10:00    674.183336
2024-12-05 13:20:00    646.762314
2024-12-05 13:30:00    555.768018
2024-12-05 13:40:00    542.663942
2024-12-05 13:50:00    523.270355
                          ...    
2025-10-16 22:10:00     66.368796
2025-10-16 22:20:00     68.900726
2025-10-16 22:30:00     64.229841
2025-10-16 22:40:00     59.253113
2025-10-16 22:50:00     61.592413
Name: Total_Surface_Area_um2_cm3, Length: 41175, dtype: float64


In [ ]:
# 将计算得到的总表面积浓度与 INP 数据合并

# 1. 准备工作：确保时间格式正确
df_inp = pd.read_csv(r"D:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.3.csv")
df_inp['Date'] = pd.to_datetime(df_inp['Date'])

# 将 Series 转换为 DataFrame，并确保索引是 datetime 类型
sa_df = total_s.to_frame().reset_index()
sa_df.columns = ['Time_A', 'Total_Surface_Area(μm2/cm3)'] # 重命名方便识别
sa_df['Time_A'] = pd.to_datetime(sa_df['Time_A'])

# 2. 关键步骤：必须进行排序
# pd.merge_asof 要求两个数据集必须按时间顺序排列
df_inp = df_inp.sort_values('Date')
sa_df = sa_df.sort_values('Time_A')

# 3. 执行模糊匹配合并
result = pd.merge_asof(
    df_inp, 
    sa_df, 
    left_on='Date',          # 左表匹配键
    right_on='Time_A',      # 右表匹配键
    direction='nearest',     # 搜索方向：最近的时间点
    tolerance=pd.Timedelta('1h') # 容差限制：1小时
)

# 4. 可选：清理辅助列
# 如果匹配失败（超过1小时），Total_Surface_Area 列会自动填充为 NaN
# Time_A 列可以用来检查实际匹配到了哪个时间点，如果不需要可以删掉
# result = result.drop(columns=['Time_A'])

In [11]:
#result = result.dropna(subset=['Total_Surface_Area(μm2/cm3)'])
print(f"合并后数据的维度: {result.shape}")
result['n_s'] = result['N_inp_net(#/L)'] / result['Total_Surface_Area(μm2/cm3)'] * 1e9 # unit: # / m^2

合并后数据的维度: (2678, 10)


In [26]:
# 保存最终的数据到 CSV 文件
final_psd_df.to_csv(r"D:\Coding\Data\Lanzhou_aerosol\SMPS+APS\final_psd(outer).csv")
result.to_csv(r"D:\Coding\Data\Lanzhou_aerosol\SMPS+APS\INP+ns(outer).csv", index=False)
status_df.to_csv(r"D:\Coding\Data\Lanzhou_aerosol\SMPS+APS\instrument_status.csv", index=False)